# Data analysis with pandas (1): tables, selection and grouping

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*X. Cid Vidal* (original material, 2025). Adapted by *J. A. Hernando Morata*.

numpy gives us fast arrays; **pandas** gives them labels, columns of different types and
the tools to select, group, combine, clean and store tabular data. Part 1: the tables and
how to select and group them.

**Objectives**

* use the two structures of pandas: **Series** and **DataFrame**, and their **index**;
* select and modify data with `.loc`, `.iloc`, masks and `query`, under **Copy-on-Write**;
* deal with **missing data** and with the dtypes of pandas 3 (`str`, `category`, `Int64`);
* compute with columns and **group** rows: `groupby`, `agg`, `transform`, `map`.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 10)

## 1. Series and DataFrame

* **Series**: a one-dimensional array with an **index** (a label for each value) and a
  name.
* **DataFrame**: a table, a collection of Series (the **columns**) that share the same
  index. Each column has its own dtype.

pandas is built on numpy: the values of a column are often a numpy array.

### 1.1 Series: an array with labels

In [ ]:
arr = np.array([10, 20, 30])
ser = pd.Series([10, 20, 30], index=['a', 'b', 'c'], name='values')
print(arr)
print(ser)

In [ ]:
print(ser['b'], ser.index.tolist(), ser.values)
print(pd.Series({'x': 1.5, 'y': 2.5}))      # from a dict: keys become the index

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Adding two Series</b></p>

**Decide** what this prints, before running it.

```python
a = pd.Series([1, 2, 3], index=['x', 'y', 'z'])
b = pd.Series([10, 20], index=['z', 'y'])
print(a + b)
```

</div>

In [ ]:
a = pd.Series([1, 2, 3], index=['x', 'y', 'z'])
b = pd.Series([10, 20], index=['z', 'y'])
print(a + b)
print(a.add(b, fill_value=0))

<details>
<summary><b>Solution</b></summary>

`x NaN, y 22.0, z 13.0`. pandas **aligns by label**, not by position: `y` is added to `y`,
`z` to `z`. `x` has no partner, so the result is `NaN` (and the dtype becomes float).
`a.add(b, fill_value=0)` treats the missing partner as `0`. With numpy arrays of
different length the sum would simply fail.

</details>

### 1.2 DataFrame: a table of columns

A DataFrame can be built from a dict of columns, a list of rows (dicts) or a 2D array:

In [ ]:
df = pd.DataFrame({'run': [1, 2, 3], 'detector': ['tracker', 'calo', 'calo']})
rows = [{'run': 4, 'detector': 'muon'}, {'run': 5, 'detector': 'calo'}]
print(df)
print(pd.DataFrame(rows))
print(pd.DataFrame(np.eye(2), columns=['u', 'v']))

### 1.3 Index types

| Index | Description | Use case |
|:--|:--|:--|
| `RangeIndex` | 0, 1, 2… (default) | simple tables |
| `DatetimeIndex` | time stamps | time series, e.g. detector logs |
| `CategoricalIndex` | a few discrete labels | e.g. particle types |
| `MultiIndex` | several levels | e.g. run × detector |

In [ ]:
print(pd.date_range('2025-10-01', periods=3, freq='h'))
mi = pd.MultiIndex.from_product([[1, 2], ['tracker', 'calo']], names=['run', 'detector'])
rates = pd.Series([10.1, 3.2, 9.8, 3.5], index=mi, name='rate')
print(rates)
print(rates.loc[2])                        # all the detectors of run 2

## 2. A dataset: particles in collision events

A synthetic table, one row per particle: the event it belongs to, its type, charge,
transverse momentum $p_T$ (GeV), pseudorapidity $\eta$ and azimuth $\phi$.

The generator lives in a small module, `fca_events.py`, next to this notebook: part 2
imports the same function instead of copying it.

In [ ]:
import inspect
from fca_events import make_events

print(inspect.getsource(make_events))

`ptype` has dtype `category`: a few repeated labels stored as small integer codes (see
4.3). `missing=0.02` sets about 2 % of the $\eta$ values to `NaN`: failed measurements.

In [ ]:
rng = np.random.default_rng(2025)
events = make_events(2000, rng, missing=0.02)
events.head()

In [ ]:
print(events.shape)
print(events.dtypes)

In [ ]:
events.describe().round(2)

***[+] Lookout.*** Real data in high-energy physics is often stored in **ROOT** files. pandas does not read
them, but [`uproot`](https://uproot.readthedocs.io) does:
`uproot.open('file.root')['tree'].arrays(library='pd')` returns a DataFrame.

## 3. Selection

Each row has a **label** (its value in the index) and a **position** (0, 1, 2…). Each
column has a label (its name) and a position. pandas selects by either:

| | by label | by position |
|:--|:--|:--|
| rows and columns | `.loc[rows, cols]` | `.iloc[rows, cols]` |
| a single value | `.at[row, col]` | `.iat[i, j]` |

In `events` the labels are 0, 1, 2…: label and position coincide and hide the difference.
A small table with text labels shows it:

In [ ]:
det = pd.DataFrame({'rate': [10.1, 3.2, 0.8, 5.5],
                    'temp': [20.1, 21.3, 19.8, 22.0],
                    'on': [True, True, False, True]},
                   index=['tracker', 'calo', 'muon', 'trigger'])
det

### 3.1 `.loc`: by label

* `det.loc[row, col]`: one value. `det.loc[row]`: one row, as a Series.
* A **list** of labels gives a DataFrame.
* A **slice** of labels **includes the end**: labels have no "next one" to stop at.
* `:` means all the rows (or columns).

In [ ]:
print(det.loc['calo', 'rate'])
print(det.loc['calo'])                            # mixed types: dtype object
det.loc['tracker':'muon', ['rate', 'temp']]       # 'muon' is included

### 3.2 `.iloc`: by position

The numpy rules: positions from 0, the end of a slice excluded, negative positions count
from the end. Labels are ignored.

In [ ]:
print(det.iloc[1, 0])                # row 1, column 0: the same value as above
det.iloc[:2, -2:]                    # first two rows, last two columns

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — After sorting</b></p>

**Predict** the three results.

```python
s = det.sort_values('rate')
s.iloc[0, 0]
s.loc['calo', 'rate']
s.loc['calo':'tracker']
```

</div>

In [ ]:
s = det.sort_values('rate')
print(s.iloc[0, 0])
print(s.loc['calo', 'rate'])
s.loc['calo':'tracker']

<details>
<summary><b>Solution</b></summary>

* `0.8`: position 0 is now `muon`, the lowest rate.
* `3.2`: the label `calo` **travels with its row**; positions do not.
* `calo`, `trigger`, `tracker`: a slice of labels follows the **current order** of the
  rows, from `calo` to `tracker`, both included.

Use `.loc` when the meaning is "this detector"; `.iloc` when it is "the first rows".

</details>

### 3.3 Integer labels: the confusing case

A selection keeps the **original labels** of the rows. After a cut, the labels of
`events` are no longer 0, 1, 2…: `.loc[0]` is the row with label 0, which may be gone;
`.iloc[0]` is the first row that is left.

In [ ]:
sel = events[events['pt'] > 50]
print(sel.index[:5].tolist())                     # labels of the first rows
print(sel.iloc[0].name, 0 in sel.index)           # label of the first row; is 0 a label?
sel.reset_index(drop=True).head(3)                # new labels 0, 1, 2…

### 3.4 `.at` and `.iat`: a single value

Same as `.loc` and `.iloc`, but they accept only one row and one column: the code says
that a single value is meant, and they are a little faster. Neither is a reason to loop
over the rows: for whole columns, use vectorized operations.

In [ ]:
print(det.at['calo', 'rate'], det.iat[1, 0])
fixed = det.copy()
fixed.at['muon', 'on'] = True                     # set one value
fixed

### 3.5 Boolean masks

A comparison on a column gives a **boolean Series**, with the same index as the table.
Used as a selector, it keeps the rows where it is `True`:

* `df[mask]`: those rows, all the columns;
* `df.loc[mask, cols]`: those rows, some columns.

In [ ]:
mask = det['rate'] > 1
print(mask)
det[mask]

Combine masks with `&` (and), `|` (or) and `~` (not), never with `and`, `or`, `not`; put
each condition between parentheses: `&` binds tighter than `>`.

In [ ]:
det.loc[det['on'] & (det['temp'] < 22), 'rate']

In [ ]:
events.loc[(events['ptype'] == 'muon') & (events['pt'] > 50), ['event', 'pt', 'eta']]

***Careful with `df[...]`.*** The square brackets alone do different things:

| | |
|:--|:--|
| `df['pt']` | a column (Series) |
| `df[['pt', 'eta']]` | some columns (DataFrame) |
| `df[mask]` | some rows |
| `df[2:5]` | rows by **position** (avoid: use `.iloc`) |

In code that others will read, `.loc` and `.iloc` say what is meant.

### 3.6 query and eval

`query` selects rows with a string expression; `eval` computes columns. `@name` refers to
a Python variable.

In [ ]:
pt_cut = 60.
events.query('pt > @pt_cut and abs(eta) < 1 and charge == 1')

In [ ]:
events.eval('pz = pt * sinh(eta)').head(3)

`eval` returns a **new** DataFrame: `events` does not get the `pz` column.

### 3.7 Setting values: Copy-on-Write

Assign in a **single step**: `df.loc[rows, cols] = value`.

Since pandas 3.0, *Copy-on-Write* is always on: the result of any selection behaves as a
**new object**. Changing it never changes the original
([user guide](https://pandas.pydata.org/docs/user_guide/copy_on_write.html)).

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Chained assignment</b></p>

**Decide**: after this code, has `ev` changed?

```python
ev = events.head(6).copy()
ev[ev['pt'] > 10]['pt'] = 0.
```

</div>

In [ ]:
ev = events.head(6).copy()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    ev[ev['pt'] > 10]['pt'] = 0.          # chained assignment
print([w.category.__name__ for w in caught])
ev['pt']

<details>
<summary><b>Solution</b></summary>

No. `ev[ev['pt'] > 10]` is a new (temporary) DataFrame, and `['pt'] = 0.` changes that
temporary object, which is then thrown away. pandas warns with a
`ChainedAssignmentError`. The right way is one step:
`ev.loc[ev['pt'] > 10, 'pt'] = 0.`

Before pandas 3 the result depended on whether the selection was a view or a copy (the
old `SettingWithCopyWarning`). Now the rule is simple: one object, one assignment.

</details>

In [ ]:
ev.loc[ev['pt'] > 10, 'pt'] = 0.         # single step: modifies ev
sub = ev[ev['charge'] > 0]
sub['pt'] = -1.                           # modifies sub only, no warning
print(ev['pt'].tolist())

## 4. Operations and missing data

### 4.1 Vectorized columns

Columns are arrays: compute with operators and numpy functions, never with a loop.
`assign` returns a new DataFrame with the new columns.

In [ ]:
kin = events.assign(px=events['pt'] * np.cos(events['phi']),
                    py=events['pt'] * np.sin(events['phi']),
                    pz=events['pt'] * np.sinh(events['eta']))
kin[['pt', 'eta', 'px', 'py', 'pz']].head(3)

In [ ]:
centered = events[['pt', 'eta']] - events[['pt', 'eta']].mean()   # column by column
centered.mean().round(10)

### 4.2 Missing data

`NaN` propagates in arithmetic and is skipped by the reductions (`mean`, `sum`…).

In [ ]:
print(events.isna().sum())
print(len(events), len(events.dropna(subset=['eta'])))
print(events['eta'].mean(), events['eta'].fillna(events['eta'].median()).mean())

In a time series, a gap can be filled with the previous value (`ffill`) or interpolated:

In [ ]:
temp = pd.Series([20.1, np.nan, np.nan, 20.7],
                 index=pd.date_range('2025-10-01', periods=4, freq='h'))
pd.DataFrame({'raw': temp, 'ffill': temp.ffill(), 'interpolate': temp.interpolate()})

### 4.3 dtypes: str, category, Int64

* Text columns have dtype **`str`** in pandas 3 (`object` before); missing text is `NaN`.
* **`category`** stores a few repeated labels as small integer codes: less memory, faster
  grouping.
* A `NaN` turns an integer column into `float`; the **nullable** `Int64` keeps integers
  and marks the gap as `<NA>`.

In [ ]:
s = pd.Series(['pion', 'kaon', 'pion'])
print(s.dtype, '|', s.astype('category').cat.codes.tolist())
print(pd.Series([1, None, -1]).dtype, '|', pd.Series([1, None, -1], dtype='Int64').tolist())

## 5. Grouping

### 5.1 Split, apply, combine

`df.groupby(key)` **splits** the rows by the values of `key`. Nothing is computed until
we **apply** a function (`mean`, `sum`, `agg`…); pandas then **combines** the results in
a table indexed by the key.

In [ ]:
events.groupby('ptype')['pt'].mean()

Several results at once with `agg` and **named aggregation**: each keyword
`new_name=(column, function)` becomes a column of the result.

In [ ]:
events.groupby('ptype').agg(n=('pt', 'size'), mean_pt=('pt', 'mean'),
                            max_pt=('pt', 'max'), net_charge=('charge', 'sum'))

### 5.2 One row per event

The same tool builds event-level variables, e.g. the multiplicity and
$H_T = \sum p_T$:

In [ ]:
per_event = events.groupby('event').agg(n=('pt', 'size'), ht=('pt', 'sum'),
                                        charge=('charge', 'sum'))
per_event.query('n >= 12')

### 5.3 transform: back to the original rows

`agg` returns one value per group; `transform` returns one value per **row**, aligned with
the original table. Example: the fraction of $H_T$ carried by each particle.

In [ ]:
frac = events['pt'] / events.groupby('event')['pt'].transform('sum')
print(frac.head(4).round(3).tolist())
print(frac.groupby(events['event']).sum().head(3).tolist())

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — The leading particle</b></p>

The **leading** particle of an event is the one with the largest $p_T$.

1. Select the leading particle of each event. Hint: `groupby('event')['pt'].idxmax()`
   gives its row label; use it with `.loc`.
2. Check that there is one row per event.
3. Compare the fraction of each type among the leading particles and among all the
   particles (`value_counts(normalize=True)`). Do you expect a difference?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
lead = events.loc[events.groupby('event')['pt'].idxmax()]
print(len(lead), events['event'].nunique())
pd.DataFrame({'leading': lead['ptype'].value_counts(normalize=True),
              'all': events['ptype'].value_counts(normalize=True)}).round(3)
```

In this generator $p_T$ does not depend on the type, so the fractions agree within the
statistical fluctuations (about 2000 leading particles). In real data they would not:
e.g. muons from heavy decays are harder than pions.

</details>

### 5.4 map, apply, transform

| Method | Use | Speed |
|:--|:--|:--|
| `Series.map(dict or f)` | element by element, e.g. relabel | fast |
| `apply(f, axis=1)` | any function, row by row | slow: a Python loop |
| `groupby(...).transform(f)` | per group, keeps the shape | medium |

In [ ]:
mass = {'pion': 0.1396, 'kaon': 0.4937, 'muon': 0.1057, 'electron': 0.000511}  # GeV
m = events['ptype'].map(mass).astype(float)
p = events['pt'] * np.cosh(events['eta'])
events.assign(E=np.sqrt(p**2 + m**2)).head(3)

`map` on a `category` column returns a `category` (if the mapping is one to one):
`astype(float)` makes it numeric.

## 6. Practice

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Event kinematics</b></p>

Using `events` and the masses `mass` of section 5.4:

1. Drop the particles without $\eta$ and add $p_x, p_y, p_z$, $p$ and $E$.
2. Per event (`groupby('event')`): multiplicity, $\sum p_T$, total charge, and the sums of
   $E, p_x, p_y, p_z$.
3. Compute the invariant mass of each event, $M^2 = (\sum E)^2 - |\sum \vec p|^2$.
4. Histogram $M$ and $\sum p_T$, and plot $M$ vs the multiplicity.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
ev = events.dropna(subset=['eta'])
mev = ev['ptype'].map(mass).astype(float)
ev = ev.assign(px=ev['pt'] * np.cos(ev['phi']), py=ev['pt'] * np.sin(ev['phi']),
               pz=ev['pt'] * np.sinh(ev['eta']))
ev = ev.assign(p=np.sqrt(ev['px']**2 + ev['py']**2 + ev['pz']**2))
ev = ev.assign(E=np.sqrt(ev['p']**2 + mev**2))
summ = ev.groupby('event').agg(n=('pt', 'size'), ht=('pt', 'sum'), q=('charge', 'sum'),
                               E=('E', 'sum'), px=('px', 'sum'), py=('py', 'sum'),
                               pz=('pz', 'sum'))
summ['M'] = np.sqrt(summ['E']**2 - summ['px']**2 - summ['py']**2 - summ['pz']**2)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].hist(summ['M'], bins=50, range=(0, 1000))
axes[0].set(xlabel='M [GeV]')
axes[1].hist(summ['ht'], bins=50)
axes[1].set(xlabel=r'$\sum p_T$ [GeV]')
axes[2].scatter(summ['n'], summ['M'], s=4)
axes[2].set(xlabel='multiplicity', ylabel='M [GeV]', yscale='log')
plt.tight_layout()
plt.show()
```

$M$ grows with the multiplicity: these are independent random particles, not the decay
products of a resonance, so there is no peak.

</details>

## 7. Summary

* A **Series** is an array with an index; a **DataFrame**, columns sharing an index.
  Operations **align by label**.
* Select with `.loc` (labels), `.iloc` (positions), masks (`&`, `|`, `~`) and `query`;
  assign in **one step** with `.loc`: under Copy-on-Write a chained assignment changes
  nothing.
* `NaN` marks missing data: `isna`, `dropna`, `fillna`, `ffill`, `interpolate`.
* dtypes of pandas 3: `str` for text, `category` for repeated labels, `Int64` for
  integers with gaps.
* Compute with whole columns (`assign`); `groupby` + `agg` (one row per group) or
  `transform` (one value per row); `map` to relabel.

<details>
<summary><b>[i] Cheat sheet — pandas (1)</b></summary>

| | |
|:--|:--|
| `pd.Series(v, index=)`, `pd.DataFrame(dict)` | build |
| `df.head()`, `df.dtypes`, `df.describe()` | inspect |
| `df.loc[rows, cols]`, `df.iloc[i, j]`, `df[mask]` | select |
| `(m1) & (m2)`, `\|`, `~` | combine masks |
| `df.query('pt > @cut')`, `df.eval('z = x + y')` | string expressions |
| `df.loc[mask, 'c'] = v` | assign (one step) |
| `df.assign(c=...)` | new DataFrame with new columns |
| `isna`, `dropna`, `fillna`, `ffill`, `interpolate` | missing data |
| `astype('category')`, `dtype='Int64'` | dtypes |
| `groupby(k).agg(name=(col, f))`, `.transform(f)` | grouping |
| `s.map(dict)`, `df.apply(f, axis=1)` | element / row functions |

</details>

### Check yourself

1. What is the result of adding two Series with different indices?
2. Why does `df[df.x > 0]['y'] = 0` not change `df`? Write the correct line.
3. Why must masks be combined with `&` and not with `and`?
4. What happens to the dtype of an integer column when a value is missing? How do you
   avoid it?
5. What is the difference between `groupby(...).agg` and `groupby(...).transform`?

<details>
<summary><b>Solutions</b></summary>

1. The values are aligned by label; labels present in only one Series give `NaN`.
2. `df[df.x > 0]` is a new object (Copy-on-Write); the assignment changes it, not `df`.
   `df.loc[df.x > 0, 'y'] = 0`.
3. `and` asks for the truth value of a whole Series, which is ambiguous (error); `&`
   combines the masks element by element. `&` binds tighter than `>`: hence the
   parentheses.
4. It becomes `float`, with `NaN` in the gap. Use the nullable dtype `Int64`.
5. `agg` gives one row per group; `transform` gives one value per original row, aligned
   with the original table.

</details>

**Next**: [part 2](pandas2.ipynb) combines and reshapes tables, plots them, reads and
writes them (CSV, HDF5) and builds a tested cleaning pipeline.